# 03 - Environment Validation

Sanity-check the `ChipTestingEnv`: observation/action spaces, sequential reveal dynamics, reward correctness and a full random rollout.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import CONFIG
from src.data.preprocessing import load_processed_data
from src.environment.chip_testing_env import ChipTestingEnv, Action

data = load_processed_data(CONFIG)
env = ChipTestingEnv(data.test, data.feature_columns, CONFIG)
print('Observation space:', env.observation_space)
print('Action space:', env.action_space)
print('Number of stages:', env.n_stages, '| features:', env.n_features)

In [ ]:
# Step through one chip, continuing then stopping.
obs, info = env.reset(options={'index': 0})
print('Reset | true label:', info['true_label'], '| revealed stages:', info['revealed_stages'])
for _ in range(2):
    obs, r, term, trunc, info = env.step(Action.CONTINUE)
    print(f'CONTINUE -> reward={r:.1f} revealed={info["revealed_stages"]} cost={info["test_cost_incurred"]:.1f}')
obs, r, term, trunc, info = env.step(Action.STOP_PASS)
print(f'STOP_PASS -> reward={r:.1f} terminated={term} predicted={info["predicted_label"]}')

In [ ]:
# Random rollout over the first 200 chips.
from src.agents.random_agent import RandomAgent
from src.evaluation.evaluate import rollout_agent
from src.evaluation.metrics import full_metrics

agent = RandomAgent(CONFIG)
res = rollout_agent(agent, env, indices=list(range(min(200, env.n_samples))))
full_metrics(res, CONFIG)